# Cross-Lingual Evaluation — terminology translation stress test

Probes the RAG pipeline with queries that ask for the official equivalent of a railway term (different language combinations). 

This is an **exploratory pipeline stress test**, not a production evaluation:
* **Mono-RAG** — retrieves German chunks only; the LLM must rely on parametric knowledge
  to produce the target-language term.
* **Multi-RAG** — restricts retrieval to the target language (`filter_language`) with
  BM25 disabled (`BM25_WEIGHT=0`), so only the multilingual embedding similarity drives
  ranking.

> **Prerequisites**: run `02_mono_rag.ipynb` and `03_multi_rag.ipynb` first to build
> both ChromaDB collections.

### Content
* Section 0: Setup
* Section 1: Run cross-lingual evaluation
* Section 2: Term match evaluation
* Section 3: Export

## Section 0 — Setup

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from nltk.stem import SnowballStemmer

from chunking   import load_documents, build_chunk_records
from retrieval  import build_bm25_index, retrieve
from generation import ask
from config     import (
    EMBED_MODEL, CE_MODEL,
    TOP_K, BM25_WEIGHT, SEMANTIC_POOL, RERANK_POOL,
    GENERATION_MODEL,
)

# Shared RAG config (identical to 05_QA_evaluation.ipynb)
CFG_MONO = {
    "TOP_K":          TOP_K,
    "BM25_WEIGHT":    BM25_WEIGHT,
    "SEMANTIC_POOL":  SEMANTIC_POOL,
    "RERANK_POOL":    RERANK_POOL,
    "MODEL_NAME":     GENERATION_MODEL,
    "COLLECTION_NAME": "fdv_de_only",
    "CHROMA_DIR":     "chroma_db_de",
    "DATA_DIRS":      {"de": Path("data/de")},
}
CFG_MULTI = {
    "TOP_K":          TOP_K,
    "BM25_WEIGHT":    BM25_WEIGHT,
    "SEMANTIC_POOL":  SEMANTIC_POOL,
    "RERANK_POOL":    RERANK_POOL,
    "MODEL_NAME":     GENERATION_MODEL,
    "COLLECTION_NAME": "fdv_multilingual",
    "CHROMA_DIR":     "chroma_db_multilingual",
    "DATA_DIRS":      {"de": Path("data/de"),
                       "fr": Path("data/fr"),
                       "it": Path("data/it")},
}

# Cross-lingual config: BM25 off + cross-encoder skipped 
cfg_cl = dict(CFG_MULTI)
cfg_cl["BM25_WEIGHT"] = 0
cfg_cl["SKIP_RERANK"] = True

QUESTIONS_FILE  = Path("cross_lingual_questions.csv")
RESULTS_EXPORT  = Path("results/crosslingual_results.csv")
Path("results").mkdir(exist_ok=True)

print(f"Generation model : {GENERATION_MODEL}")
print(f"BM25_WEIGHT mono : {CFG_MONO['BM25_WEIGHT']}")
print(f"BM25_WEIGHT multi: {cfg_cl['BM25_WEIGHT']}")
print(f"SKIP_RERANK multi: {cfg_cl['SKIP_RERANK']}")

Generation model : gemma3:4b
BM25_WEIGHT mono : 0.4
BM25_WEIGHT multi: 0
SKIP_RERANK multi: True


In [2]:
# Load shared models
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print("Loading cross-encoder...")
cross_encoder = CrossEncoder(CE_MODEL)

def _load_rag_system(cfg):
    docs = []
    for lang, data_dir in cfg["DATA_DIRS"].items():
        docs.extend(load_documents(data_dir, lang))
    records = build_chunk_records(docs)
    bm25    = build_bm25_index(records)
    client  = chromadb.PersistentClient(path=str(cfg["CHROMA_DIR"]))
    coll    = client.get_or_create_collection(
        name=cfg["COLLECTION_NAME"],
        metadata={"hnsw:space": "cosine"},
    )
    assert coll.count() > 0, (
        f"Collection '{cfg['COLLECTION_NAME']}' is empty — "
        "run mono_rag.ipynb / multi_rag.ipynb first."
    )
    assert coll.count() == len(records), (
        f"ChromaDB has {coll.count()} chunks but chunk_records has {len(records)}. "
        "Delete the chroma_db_* folder and re-run mono_rag.ipynb / multi_rag.ipynb."
    )
    print(f"  {cfg['COLLECTION_NAME']}: {coll.count()} chunks, {len(records)} records")
    return records, bm25, coll

print("\nLoading mono_rag...")
chunks_mono, bm25_mono, coll_mono = _load_rag_system(CFG_MONO)
print("Loading multi_rag...")
chunks_multi, bm25_multi, coll_multi = _load_rag_system(CFG_MULTI)
print("\nBoth systems ready.")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6637.30it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5677.79it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading mono_rag...
  fdv_de_only: 1360 chunks, 1360 records
Loading multi_rag...
  fdv_multilingual: 4272 chunks, 4272 records

Both systems ready.


In [13]:
# Load cross-lingual benchmark
df_cl = pd.read_csv(QUESTIONS_FILE)
df_cl.columns = [c.strip() for c in df_cl.columns]
print(f"{len(df_cl)} cross-lingual questions loaded.")
display(df_cl)

18 cross-lingual questions loaded.


,ID,Source_Term,Source_Language,Query,Target_Language,Gold_Term,Relevant_Section
0,CL-01,Abstossen,de,Wie lautet der offizielle französische Fachbeg...,fr,lancer,"R 300.4, section 1.3"
1,CL-02,Lokführer,de,Welche offizielle Abkürzung wird auf Französis...,fr,MEC,"R 300.13, section 3.2.6"
2,CL-03,Zwergsignal,de,Wie heissen Zwergsignale auf Französisch?,fr,signaux nains,"R 300.2, section 1.2.1"
3,CL-04,Gefälle,de,Wie lautet der französische Fachbegriff für Ge...,fr,pente,"R 300.14, section 2.3.7"
4,CL-05,Einfahrweiche,de,Wie wird die Einfahrweiche auf Französisch bez...,fr,aiguille d'entrée,"R 300.4, section 4.3.2"
5,CL-06,Perron,de,Welchen Begriff wird auf Französisch für Perro...,fr,quai,"R 300.4, section 2.3.2"
6,CL-07,Abstossen,de,Wie lautet der offizielle italienische Fachbeg...,it,colpo,"R 300.4, section 1.3"
7,CL-08,Lokführer,de,Welche offizielle Abkürzung wird auf Italienis...,it,MAC,"R 300.13, section 3.2.6"
8,CL-09,Zwergsignal,de,Wie heissen Zwergsignale auf Italienisch?,it,segnali bassi,"R 300.2, section 1.2.1"
9,CL-10,Kreuzung,de,Wie lautet der italienische Fachbegriff für Kr...,it,incrocio,"R 300.6, section 1.3.4"


In [14]:
# _stem_match helper (copied from 05_QA_evaluation.ipynb)
# lang is the Target_Language — the language the gold term is written in.
LANG_TO_SNOWBALL = {"de": "german", "fr": "french", "it": "italian"}

def _stem_match(answer, term, lang):
    """All content stems of term must appear somewhere in the stemmed answer."""
    stemmer = SnowballStemmer(LANG_TO_SNOWBALL.get(lang, "english"))
    def stems(text):
        tokens = re.sub(r"[^\w\s]", " ", str(text).lower()).split()
        return {stemmer.stem(t) for t in tokens if len(t) >= 4}
    target_stems = stems(term)
    if not target_stems:
        return str(term).lower() in str(answer).lower()
    return target_stems.issubset(stems(answer))

## Section 1 — Run cross-lingual evaluation

* **Mono-RAG**: default cfg (`BM25_WEIGHT=0.4`, cross-encoder on), no `filter_language`. Retrieves German chunks; the LLM must rely on parametric knowledge to produce the target-language term.
* **Multi-RAG**: `cfg_cl` (`BM25_WEIGHT=0`, `SKIP_RERANK=True`), `filter_language=Target_Language`. Retrieves only target-language chunks ranked by pure cosine similarity (Pass 1 only). BM25 and the cross-encoder are both disabled because they reward token overlap, which systematically promotes generic "FDV/PCT" admin chunks over the specific terminology sections we need.

In [15]:
eval_rows = []
total = len(df_cl)

for i, row in df_cl.iterrows():
    print(f"  [{i+1}/{total}] {row['ID']}  {row['Query'][:70]}", end="\r")

    # Mono-RAG: no language filter, standard BM25 hybrid
    ans_mono, chunks_mono_ret = ask(
        row["Query"], embedder, cross_encoder,
        coll_mono, bm25_mono, chunks_mono, CFG_MONO,
    )

    # Multi-RAG: target-language filter, BM25 disabled
    ans_multi, chunks_multi_ret = ask(
        row["Query"], embedder, cross_encoder,
        coll_multi, bm25_multi, chunks_multi, cfg_cl,
        filter_language=row["Target_Language"],
    )

    eval_rows.append({
        "ID":               row["ID"],
        "Source_Term":      row["Source_Term"],
        "Source_Language":  row["Source_Language"],
        "Target_Language":  row["Target_Language"],
        "Gold_Term":        row["Gold_Term"],
        "Relevant_Section": row["Relevant_Section"],
        "query":            row["Query"],
        "answer_mono":      ans_mono,
        "answer_multi":     ans_multi,
        "ref_mono":         chunks_mono_ret[0]["text"]  if chunks_mono_ret  else "",
        "ref_multi":        chunks_multi_ret[0]["text"] if chunks_multi_ret else "",
    })

df_results = pd.DataFrame(eval_rows)
print(f"\nEvaluation complete — {len(df_results)} rows.")
display(df_results[["ID", "Source_Term", "Target_Language", "Gold_Term",
                     "answer_mono", "answer_multi"]].head(6))

  [18/18] CL-18  Quel est le terme allemand équivalent à chantiers????en Lokführer ver
Evaluation complete — 18 rows.


,ID,Source_Term,Target_Language,Gold_Term,answer_mono,answer_multi
0,CL-01,Abstossen,fr,lancer,Der französische Fachbegriff für Abstossen ist...,"Laut Abschnitt 7.2.7, PCT, ist der offizielle ..."
1,CL-02,Lokführer,fr,MEC,Die Antwort auf diese Frage lässt sich aus dem...,"Laut Abschnitt 2.5, Sektion 2.5 der FDV (PCT) ..."
2,CL-03,Zwergsignal,fr,signaux nains,Die Frage kann anhand der bereitgestellten Aus...,Die Frage kann anhand des bereitgestellten Tex...
3,CL-04,Gefälle,fr,pente,Laut PCT (Prescriptions de circulation des tra...,Die bereitgestellten Auszüge enthalten keine I...
4,CL-05,Einfahrweiche,fr,aiguille d'entrée,Laut Abschnitt 2.5 der Grundlagen wird die Ein...,Die Antwort lässt sich aus dem bereitgestellte...
5,CL-06,Perron,fr,quai,Die Antwort lässt sich aus den bereitgestellte...,"Laut Abschnitt 8.3.1, Absatz 1 der FDV wird fü..."


## Section 2 — Term match evaluation

Four boolean columns per row:
* **exact_mono / exact_multi** — `Gold_Term` (case-insensitive) is a substring of the answer.
* **stem_mono / stem_multi** — Snowball stemmer collapses inflections before matching.
  The stemmer language is `Target_Language` because the gold term is in the target language.

In [16]:
df_results["exact_mono"]  = df_results.apply(
    lambda r: str(r["Gold_Term"]).lower() in str(r["answer_mono"]).lower(), axis=1)
df_results["exact_multi"] = df_results.apply(
    lambda r: str(r["Gold_Term"]).lower() in str(r["answer_multi"]).lower(), axis=1)
df_results["stem_mono"]   = df_results.apply(
    lambda r: _stem_match(r["answer_mono"],  r["Gold_Term"], r["Target_Language"]), axis=1)
df_results["stem_multi"]  = df_results.apply(
    lambda r: _stem_match(r["answer_multi"], r["Gold_Term"], r["Target_Language"]), axis=1)

n = len(df_results)

# ── Overall summary ───────────────────────────────────────────────────────────
print(f"{'':22} {'mono_rag':>10} {'multi_rag':>10}")
print("-" * 44)
for label, mc, mlc in [
    ("Exact match",    "exact_mono", "exact_multi"),
    ("Stem match",     "stem_mono",  "stem_multi"),
]:
    m  = df_results[mc].mean()
    ml = df_results[mlc].mean()
    print(f"{label:22} {100*m:>9.1f}% {100*ml:>9.1f}%")

# ── By target language ────────────────────────────────────────────────────────
lang_r = df_results.groupby("Target_Language")[
    ["exact_mono", "exact_multi", "stem_mono", "stem_multi"]
].mean() * 100
lang_r.columns = ["exact_mono(%)", "exact_multi(%)", "stem_mono(%)", "stem_multi(%)"]
print("\nBy target language:")
display(lang_r.round(1))

# ── By direction ─────────────────────────────────────────────────────────────
df_results["direction"] = df_results["Source_Language"] + "→" + df_results["Target_Language"]
dir_r = df_results.groupby("direction")[
    ["exact_mono", "exact_multi", "stem_mono", "stem_multi"]
].mean() * 100
dir_r.columns = ["exact_mono(%)", "exact_multi(%)", "stem_mono(%)", "stem_multi(%)"]
print("\nBy direction:")
display(dir_r.round(1))

                         mono_rag  multi_rag
--------------------------------------------
Exact match                  0.0%       5.6%
Stem match                   0.0%       5.6%

By target language:


,exact_mono(%),exact_multi(%),stem_mono(%),stem_multi(%)
Target_Language,,,,
de,0.0,0.0,0.0,0.0
fr,0.0,0.0,0.0,0.0
it,0.0,16.7,0.0,16.7



By direction:


,exact_mono(%),exact_multi(%),stem_mono(%),stem_multi(%)
direction,,,,
de→fr,0.0,0.0,0.0,0.0
de→it,0.0,16.7,0.0,16.7
fr→de,0.0,0.0,0.0,0.0


## Section 3 — Export

**Interpretation note**: mono-RAG produces answers based entirely on LLM parametric
knowledge because its index contains no FR/IT chunks — the retrieved German passage
is used as context but cannot supply the target-language term directly.
Multi-RAG retrieves the official target-language passage with BM25 disabled; any term
match therefore reflects the LLM reading the correct passage rather than recalling
from training.

In [17]:
df_results.to_csv(RESULTS_EXPORT, index=False, encoding="utf-8-sig")
print(f"Full results saved to {RESULTS_EXPORT}")
display(df_results.head(3))

Full results saved to results\crosslingual_results.csv


,ID,Source_Term,Source_Language,Target_Language,Gold_Term,Relevant_Section,query,answer_mono,answer_multi,ref_mono,ref_multi,exact_mono,exact_multi,stem_mono,stem_multi,direction
0,CL-01,Abstossen,de,fr,lancer,"R 300.4, section 1.3",Wie lautet der offizielle französische Fachbeg...,Der französische Fachbegriff für Abstossen ist...,"Laut Abschnitt 7.2.7, PCT, ist der offizielle ...",3.2.2\tAbstossen\n\t\n\tBeim Abstossen hat der...,yen du texte : « R ... » ou\n–\tde manière opt...,False,False,False,False,de→fr
1,CL-02,Lokführer,de,fr,MEC,"R 300.13, section 3.2.6",Welche offizielle Abkürzung wird auf Französis...,Die Antwort auf diese Frage lässt sich aus dem...,"Laut Abschnitt 2.5, Sektion 2.5 der FDV (PCT) ...",Schweizerische Eisenbahnen\tR 300.13\n\n\n\nLo...,sponsable de l’application des mesures de sécu...,False,False,False,False,de→fr
2,CL-03,Zwergsignal,de,fr,signaux nains,"R 300.2, section 1.2.1",Wie heissen Zwergsignale auf Französisch?,Die Frage kann anhand der bereitgestellten Aus...,Die Frage kann anhand des bereitgestellten Tex...,\n\t\n\tAuf der Rückseite der Zwergsignale wei...,autres signaux\n\t\t\tun signal avancé peut pr...,False,False,False,False,de→fr
